# Modèle 2 — Random Forest Regression
**Projet ML 2026 — UMONS | Groupe 3**

Random Forest est un algorithme d'ensemble qui combine plusieurs arbres de décision.
Chaque arbre est entraîné sur un sous-ensemble aléatoire des données et des features.
La prédiction finale est la moyenne de tous les arbres.

In [16]:
# Imports et configuration
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Import des fonctions utilitaires du projet
sys.path.append("../src")
from utils import compute_rmse, print_cv_results, create_submission

pd.set_option('display.float_format', '{:.4f}'.format)

# Chargement des données finales (nettoyées et fusionnées dans 01_EDA.ipynb)
train = pd.read_csv("../data/train_final.csv")
test  = pd.read_csv("../data/test_final.csv")

print(f"Train : {train.shape}")
print(f"Test  : {test.shape}")

Train : (1559, 67)
Test  : (669, 66)


In [17]:
# Séparation features / cible
y = train["Ja in Prozent"]

# One Hot Encoding sur Kanton
train_encoded = pd.get_dummies(train, columns=["Kanton"])
test_encoded  = pd.get_dummies(test,  columns=["Kanton"])

# Supprimer les colonnes inutiles
X = train_encoded.drop(columns=["Ja in Prozent", "Gemeinde", "commune_id", "Kantons-Nummer"])
X_test = test_encoded.drop(columns=["Gemeinde", "commune_id", "Kantons-Nummer"])

# Aligner les colonnes train et test
X_test = X_test.reindex(columns=X.columns, fill_value=0)

print(f" X train : {X.shape}")
print(f" X test  : {X_test.shape}")
print(f" y train : {y.shape}")

 X train : (1559, 88)
 X test  : (669, 88)
 y train : (1559,)


## Modèle Random Forest de base
On commence par un modèle de base avec les hyperparamètres par défaut
pour avoir une première idée des performances.

In [18]:
# Modèle de base avec hyperparamètres par défaut
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("rf", RandomForestRegressor(n_estimators=100, random_state=42))
])

# Cross-validation 5 folds
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(
    pipeline, X, y,
    cv=kf,
    scoring="neg_root_mean_squared_error"
)

rmse_scores = -scores
print("=== Résultats Cross-Validation (5 folds) ===")
print(f"RMSE par fold : {rmse_scores.round(4)}")
print(f"RMSE moyen    : {rmse_scores.mean():.4f}")
print(f"RMSE std      : {rmse_scores.std():.4f}")

=== Résultats Cross-Validation (5 folds) ===
RMSE par fold : [6.5206 6.4156 6.4032 6.3045 6.4189]
RMSE moyen    : 6.4126
RMSE std      : 0.0686


##  Optimisation des hyperparamètres
On utilise GridSearchCV pour trouver la meilleure combinaison d'hyperparamètres.

In [19]:
from sklearn.model_selection import GridSearchCV

# Grille d'hyperparamètres à tester
param_grid = {
    "rf__n_estimators": [100, 200, 300],
    "rf__max_depth": [None, 10, 20, 30],
    "rf__min_samples_split": [2, 5, 10],
    "rf__max_features": ["sqrt", "log2", 0.5]
}

pipeline_gs = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("rf", RandomForestRegressor(random_state=42))
])

grid_search = GridSearchCV(
    pipeline_gs,
    param_grid,
    cv=kf,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X, y)

print(f"\nMeilleurs hyperparamètres : {grid_search.best_params_}")
print(f"Meilleur RMSE CV : {-grid_search.best_score_:.4f}")

Fitting 5 folds for each of 108 candidates, totalling 540 fits

Meilleurs hyperparamètres : {'rf__max_depth': 20, 'rf__max_features': 0.5, 'rf__min_samples_split': 2, 'rf__n_estimators': 300}
Meilleur RMSE CV : 6.2429


## Entraînement du modèle final
On entraîne le modèle Random Forest avec les meilleurs hyperparamètres trouvés.

In [20]:
# Entraînement du modèle final
pipeline_final = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("rf", RandomForestRegressor(
        n_estimators=300,
        max_depth=20,
        max_features=0.5,
        min_samples_split=2,
        random_state=42
    ))
])

pipeline_final.fit(X, y)

# Evaluation sur le train complet
y_pred_train = pipeline_final.predict(X)
rmse_train = compute_rmse(y, y_pred_train)

print(f"Modèle Random Forest entraîné")
print(f"RMSE train complet    : {rmse_train:.4f}")
print(f"RMSE cross-validation : {6.2429:.4f}")

Modèle Random Forest entraîné
RMSE train complet    : 2.2691
RMSE cross-validation : 6.2429


## 🔧 Optimisation v2 - Correction de l'overfitting
Le modèle v1 montre de l'overfitting (RMSE train=2.27 vs RMSE CV=6.24).
On relance la recherche avec des hyperparamètres plus restrictifs.

In [21]:
# Entraînement du modèle final v2
pipeline_final_v2 = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("rf", RandomForestRegressor(
        n_estimators=500,
        max_depth=15,
        max_features=0.3,
        min_samples_split=5,
        random_state=42
    ))
])

pipeline_final_v2.fit(X, y)

y_pred_train = pipeline_final_v2.predict(X)
rmse_train   = compute_rmse(y, y_pred_train)

print(f"Modèle Random Forest v2 entraîné")
print(f"RMSE train complet    : {rmse_train:.4f}")
print(f"RMSE cross-validation : 6.2370")

Modèle Random Forest v2 entraîné
RMSE train complet    : 2.6276
RMSE cross-validation : 6.2370


## Prédictions et soumission Kaggle

In [22]:
# Prédictions sur le test
y_pred_test = pipeline_final_v2.predict(X_test)

# Création du fichier de soumission
create_submission(
    commune_ids=test["commune_id"],
    y_pred=y_pred_test,
    filename="../submissions/submission_RF_v1.csv"
)

✅ Soumission créée : ../submissions/submission_RF_v1.csv (669 lignes)


,Id,Predicted
0,69,46.6370
1,981,42.0751
2,5432,36.3657
3,387,45.5518
4,2194,32.3708
...,...,...
664,4080,40.8914
665,5425,40.8923
666,5199,47.5445
667,4251,33.7394
